In [5]:
import ee
import pandas as pd
import os
import datetime

def authenticate_and_initialize():
    """Authenticates and initializes the Earth Engine API with the specified project."""
    try:
        ee.Initialize(project='enhanced-bonito-457316-v6')
        print("Google Earth Engine initialized successfully.")
    except ee.EEException as e:
        print(f"Google Earth Engine initialization failed: {e}")
        return False
    return True

def maskL8SR(image):
    """
    Applies cloud and cloud shadow mask to a Landsat 8 SR image.
    Function from the Earth Engine Community Tutorials.
    """
    # Bit 3 is cloud shadow, bit 5 is cloud.
    cloudShadowBitMask = (1 << 3)
    cloudsBitMask = (1 << 5)

    # Get the pixel QA band.
    qa = image.select('QA_PIXEL')

    # Both flags should be set to zero, indicating clear conditions.
    mask = qa.bitwiseAnd(cloudShadowBitMask).eq(0).And(qa.bitwiseAnd(cloudsBitMask).eq(0))

    # Return the masked image.
    return image.updateMask(mask).divide(10000).copyProperties(image, ["system:time_start"])

def get_monthly_water_area(year, month, geometry):
    """
    Calculates the water area for a given month and year using Landsat 8 Collection 2.

    Args:
        year (int): The year to process.
        month (int): The month to process (1-12).
        geometry (ee.Geometry.Polygon): The study area geometry.

    Returns:
        float: The calculated water area in square kilometers.
    """
    # Create start and end dates for the month
    start_date = ee.Date.fromYMD(year, month, 1)
    end_date = start_date.advance(1, 'month')

    # Define the Landsat 8 Collection 2 image collection.
    # Filter by date and geometry, and apply cloud masking.
    landsat_collection = (
        ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
        .filterDate(start_date, end_date)
        .filterBounds(geometry)
        .map(maskL8SR)  # Use the new masking function
    )

    # Use the median composite to get a single, representative image for the month.
    # Check if the collection is empty before computing the median.
    median_composite = landsat_collection.median()
    if median_composite.bandNames().getInfo() == []:
        return 0.0

    # Calculate MNDWI (Modified Normalized Difference Water Index)
    # The formula is (Green - SWIR1) / (Green + SWIR1)
    # Green = SR_B3, SWIR1 = SR_B6 for Landsat 8 Collection 2
    mndwi = median_composite.normalizedDifference(['SR_B3', 'SR_B6']).rename('MNDWI')
    
    # Apply a threshold to classify water and non-water pixels.
    # MNDWI values greater than 0 are typically classified as water.
    water_mask = mndwi.gt(0)

    # Calculate the area of the water mask in square meters.
    water_stats = water_mask.multiply(ee.Image.pixelArea()).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=geometry,
        scale=30,
        maxPixels=1e13
    )

    # Get the area and convert it to square kilometers.
    water_area_sq_m = water_stats.get('MNDWI').getInfo()
    if water_area_sq_m is not None:
        water_area_sq_km = water_area_sq_m / 1e6
        return water_area_sq_km
    return 0.0

def process_data_and_save(start_year, end_year, output_folder):
    """
    Processes the monthly time series river data and saves it to a CSV file.

    Args:
        start_year (int): The starting year for data collection.
        end_year (int): The ending year for data collection.
        output_folder (str): The folder to save the output CSV.
    """
    # Define a smaller geometry for the Ganges River near Haridwar to reduce memory usage.
    # Haridwar coordinates: 29.9457° N, 78.1642° E
    ganges_river_geometry = ee.Geometry.Polygon([
        [78.15, 29.95],
        [78.18, 29.95],
        [78.18, 29.92],
        [78.15, 29.92]
    ])

    all_data = []
    print("Starting data collection for each month...")
    
    current_year = datetime.datetime.now().year
    current_month = datetime.datetime.now().month

    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            # Stop processing if we reach the current month of the current year
            if year > current_year or (year == current_year and month > current_month):
                print("Reached current date. Stopping data collection.")
                break
            
            # Format the date string for the row
            date_str = f"{year}-{month:02d}-01"

            try:
                # Get the river water area for the month
                water_area = get_monthly_water_area(year, month, ganges_river_geometry)
                print(f"Processed {date_str}: Calculated water area = {water_area:.2f} sq km")
                
                # Append the data to the list
                all_data.append({
                    'date': date_str,
                    'river_water_area_sqkm': water_area
                })
            except Exception as e:
                print(f"Error processing {date_str}: {e}")
                continue
        # Break out of the outer loop as well if the inner loop broke
        if year >= current_year and month >= current_month:
            break

    # Create a DataFrame from the collected data
    df = pd.DataFrame(all_data)

    # Save the data to a new CSV file
    output_filename = 'river_data_timeseries_monthly.csv'
    output_path = os.path.join(output_folder, 'river', output_filename)

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f"Data successfully saved to {output_path}")

if __name__ == '__main__':
    # Make sure to change this path to match your system.
    desktop_path = os.path.join(os.path.expanduser('~'), 'Desktop')
    base_folder = os.path.join(desktop_path, 'anjori', 'SatarkMitra-Disaster-Management-System', 'data')

    if authenticate_and_initialize():
        process_data_and_save(
            start_year=2018,
            end_year=2025,
            output_folder=base_folder
        )

Google Earth Engine initialized successfully.
Starting data collection for each month...
Processed 2018-01-01: Calculated water area = 0.73 sq km
Processed 2018-02-01: Calculated water area = 0.56 sq km
Processed 2018-03-01: Calculated water area = 0.56 sq km
Processed 2018-04-01: Calculated water area = 0.40 sq km
Processed 2018-05-01: Calculated water area = 0.47 sq km
Processed 2018-06-01: Calculated water area = 1.01 sq km
Processed 2018-07-01: Calculated water area = 1.73 sq km
Processed 2018-08-01: Calculated water area = 1.80 sq km
Processed 2018-09-01: Calculated water area = 1.42 sq km
Processed 2018-10-01: Calculated water area = 1.18 sq km
Processed 2018-11-01: Calculated water area = 0.28 sq km
Processed 2018-12-01: Calculated water area = 0.63 sq km
Processed 2019-01-01: Calculated water area = 0.00 sq km
Processed 2019-02-01: Calculated water area = 0.80 sq km
Processed 2019-03-01: Calculated water area = 0.63 sq km
Processed 2019-04-01: Calculated water area = 0.76 sq km

In [7]:
import ee
import json
import csv

try:
    # 1. Initialize the Earth Engine connection with your project ID.
    ee.Initialize(project='enhanced-bonito-457316-v6')

    # 2. Define the region of interest for Uttarakhand.
    uttarakhand_roi = ee.Geometry.BBox(77.5, 29.8, 81.0, 31.5)

    # 3. Define the year range for analysis.
    start_year = 2018
    end_year = 2025

    # --- Digital Elevation & Drainage ---
    srtm = ee.Image('USGS/SRTMGL1_003').clip(uttarakhand_roi)
    terrain = ee.Algorithms.Terrain(srtm)
    slope = terrain.select('slope')
    
    avg_elevation = srtm.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=uttarakhand_roi,
        scale=30,
        maxPixels=1e9
    ).get('elevation').getInfo()
    
    avg_slope = slope.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=uttarakhand_roi,
        scale=30,
        maxPixels=1e9
    ).get('slope').getInfo()

    # --- Land Use Analysis ---
    landcover = ee.Image('ESA/WorldCover/v200/2021').clip(uttarakhand_roi)
    
    # Corrected: Use frequencyHistogram and get the values directly
    landcover_stats = landcover.reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=uttarakhand_roi,
        scale=10,
        maxPixels=1e10
    ).get('Map').getInfo()
    
    # --- Time-Series Analysis (2018-2025) ---
    # Corrected: Switched to the recommended harmonized Sentinel-2 dataset
    sentinel2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    
    river_images = sentinel2.filterBounds(uttarakhand_roi) \
        .filterDate(f'{start_year}-01-01', f'{end_year}-12-31')
        
    # --- Print Summary and Export Data ---
    print("--- Uttarakhand River Data Summary ---")
    print(f"Time Period: {start_year} - {end_year}")
    print("\n--- Digital Elevation & Drainage ---")
    print(f"Average Elevation: {avg_elevation:.2f} meters")
    print(f"Average Slope: {avg_slope:.2f} degrees")
    
    print("\n--- Land Use/Land Cover Distribution ---")
    # Corrected: The values are now simple numbers, not dictionaries
    total_pixels = sum(landcover_stats.values())
    
    class_map = {
        '10': 'Tree cover', '20': 'Shrubland', '30': 'Grassland', '40': 'Cropland',
        '50': 'Aquatic vegetation', '60': 'Barren/sparse', '70': 'Urban/built-up',
        '80': 'Snow/ice', '90': 'Permanent water', '95': 'Herbaceous wetland',
        '100': 'Mangroves', '200': 'Moss/lichen'
    }
    
    if total_pixels > 0:
      for key, count in landcover_stats.items():
          percentage = (count / total_pixels) * 100
          description = class_map.get(key, f'Unknown ({key})')
          print(f"  - {description}: {percentage:.2f}%")
        
    print("\n--- Infrastructure (Note: Manual analysis recommended) ---")
    print("For infrastructure like roads, you would typically use OpenStreetMap data via GEE's vector assets.")

except ee.EEException as e:
    print(f"An Earth Engine error occurred. Please check your authentication and code: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

--- Uttarakhand River Data Summary ---
Time Period: 2018 - 2025

--- Digital Elevation & Drainage ---
Average Elevation: 3238.92 meters
Average Slope: 24.14 degrees

--- Land Use/Land Cover Distribution ---
  - Tree cover: 35.38%
  - Mangroves: 6.78%
  - Shrubland: 0.17%
  - Grassland: 19.43%
  - Cropland: 3.10%
  - Aquatic vegetation: 0.75%
  - Barren/sparse: 25.65%
  - Urban/built-up: 8.40%
  - Snow/ice: 0.35%
  - Permanent water: 0.01%

--- Infrastructure (Note: Manual analysis recommended) ---
For infrastructure like roads, you would typically use OpenStreetMap data via GEE's vector assets.


In [8]:
import ee
import pandas as pd
import os

def authenticate_and_initialize():
    """Authenticates and initializes the Earth Engine API with the specified project."""
    try:
        ee.Initialize(project='enhanced-bonito-457316-v6')
        print("Google Earth Engine initialized successfully.")
    except ee.EEException as e:
        print(f"Google Earth Engine initialization failed: {e}")
        return False
    return True

def get_study_area_params(geometry):
    """
    Calculates various geographical parameters for the study area.

    Args:
        geometry (ee.Geometry.Polygon): The study area geometry.

    Returns:
        dict: A dictionary containing the calculated parameters.
    """
    # 1. Elevation and Slope
    print("Extracting elevation and slope data...")
    try:
        # Use a high-resolution global elevation dataset
        elevation_image = ee.Image('USGS/SRTMGL1_003').select('elevation')
        slope_image = ee.Terrain.slope(elevation_image)
        
        # Calculate mean elevation
        mean_elevation = elevation_image.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=geometry,
            scale=30,
            maxPixels=1e13
        ).get('elevation').getInfo()
        
        # Calculate mean slope
        mean_slope = slope_image.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=geometry,
            scale=30,
            maxPixels=1e13
        ).get('slope').getInfo()

    except Exception as e:
        print(f"Error extracting elevation/slope: {e}")
        mean_elevation, mean_slope = None, None

    # 2. Land Use/Cover
    print("Extracting land use/cover data...")
    try:
        # Use the ESA WorldCover dataset for 2021
        land_cover_image = ee.Image('ESA/WorldCover/v200/2021').select('Map')

        # Compute the area of each land cover type within the geometry
        land_cover_area = ee.Image.pixelArea().addBands(land_cover_image).reduceRegion(
            reducer=ee.Reducer.sum().group(groupField=1, groupName='classification'),
            geometry=geometry,
            scale=10,
            maxPixels=1e13
        ).getInfo()['groups']

        # Format land cover data into a dictionary of percentages
        total_area_sqm = sum(d['sum'] for d in land_cover_area)
        land_cover_percentages = {}
        for group in land_cover_area:
            land_cover_id = group['classification']
            area_sqm = group['sum']
            percentage = (area_sqm / total_area_sqm) * 100
            land_cover_percentages[f'land_cover_class_{land_cover_id}_percent'] = percentage

    except Exception as e:
        print(f"Error extracting land cover: {e}")
        land_cover_percentages = {}

    return {
        'mean_elevation_meters': mean_elevation,
        'mean_slope_degrees': mean_slope,
        **land_cover_percentages
    }

def process_and_save_params(output_folder):
    """
    Processes and saves the geographical parameters to a CSV file.

    Args:
        output_folder (str): The folder to save the output CSV.
    """
    # Define the study area geometry (Ganges River near Haridwar)
    ganges_river_geometry = ee.Geometry.Polygon([
        [78.15, 29.95],
        [78.18, 29.95],
        [78.18, 29.92],
        [78.15, 29.92]
    ])

    # Get the parameters
    params_dict = get_study_area_params(ganges_river_geometry)

    # Convert the single dictionary to a DataFrame
    df = pd.DataFrame([params_dict])

    # Save the data to a new CSV file
    output_filename = 'static_river_parameters.csv'
    output_path = os.path.join(output_folder, 'river', output_filename)

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f"Static parameters successfully saved to {output_path}")

if __name__ == '__main__':
    # Make sure to change this path to match your system.
    desktop_path = os.path.join(os.path.expanduser('~'), 'Desktop')
    base_folder = os.path.join(desktop_path, 'anjori', 'SatarkMitra-Disaster-Management-System', 'data')

    if authenticate_and_initialize():
        process_and_save_params(output_folder=base_folder)

Google Earth Engine initialized successfully.
Extracting elevation and slope data...
Extracting land use/cover data...
Static parameters successfully saved to C:\Users\Dell\Desktop\anjori\SatarkMitra-Disaster-Management-System\data\river\static_river_parameters.csv
